# Phase 4 — the DPO arm (preference learning, paired lineage)

**Runtime: L4 or A100** (dev-slice work only — hardware free). ~25 min/seed on
L4, roughly half on A100. ~1–1.5 h and ~5–8 units for all three seeds.

**What DPO is, in one line:** instead of showing the model correct answers
(SFT) or letting it try-and-get-rewarded (GRPO), we show it PAIRS — "this fix
passed the tests, that attempt failed" — and nudge it toward the passing style.

**What this notebook does:**
1. **Mines the pairs for free** from the routing pass's stored samples (the
   4,352 base-model attempts saved in `phase2/routing_samples_ckpt.json`):
   re-grades every attempt by execution, then per bug builds
   chosen = a passing attempt / rejected = a failing attempt (RL pile), or
   chosen = gold fix / rejected = a failing attempt (SFT pile, all 8 fail).
   Reports a **length-bias check** (DPO's classic failure mode).
2. Per **Amendment A3 (paired lineage)**: for each seed, merges that seed's SFT
   adapter into full weights, adds a fresh LoRA, and DPO-trains with the merged
   SFT model as the reference anchor (β=0.1, 1 epoch).
3. Dev-evals + restraint-probes each seed; final cell prints SFT vs DPO side
   by side against the cross-seed noise ruler.

**Honest limitation (logged):** pairs are OFF-policy — sampled by the base
model, while the policy starts from SFT. Standard offline DPO; noted for the
write-up.

In [ ]:
# Setup: Drive, fresh repo, data, routing, stored samples
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib, gc
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
PHASE4 = '/content/drive/MyDrive/rl-post-training/phase4'
os.makedirs(PHASE4, exist_ok=True)
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
routing = {r['id']: r['pile'] for r in json.load(open(f'{PHASE2}/routing_v0.json'))}
samples = json.load(open(f'{PHASE2}/routing_samples_ckpt.json'))
train_bugs = [b for b in bugs if b['split'] == 'train']
dev_bugs = [b for b in bugs if b['split'] == 'dev']
dev_clean = [r for r in restraint if r['split'] == 'dev']
print(len(train_bugs), 'train |', len(dev_bugs), 'dev |', len(samples), 'bugs with stored attempts')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# MINE THE PAIRS: re-grade all stored attempts by execution, then pair per bug
import subprocess, tempfile, re
from concurrent.futures import ThreadPoolExecutor

def passes(code, rec, timeout=6):
    prog = '\n'.join(list(rec['test_imports']) + [code] + list(rec['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

PAIRS_PATH = f'{PHASE4}/pairs_v0.json'
if os.path.exists(PAIRS_PATH):
    pairs = json.load(open(PAIRS_PATH))
    print('pairs already mined:', len(pairs))
else:
    by_id = {b['id']: b for b in train_bugs}
    rng = random.Random(0)
    pairs = []
    done = 0
    for bid, attempts in samples.items():
        bug = by_id[bid]
        with ThreadPoolExecutor(max_workers=4) as pool:
            flags = list(pool.map(lambda c: passes(c, bug), attempts))
        good = [a for a, f in zip(attempts, flags) if f and a.strip()]
        bad = [a for a, f in zip(attempts, flags) if not f and a.strip()]
        if not bad:
            done += 1; continue   # easy pile: nothing to reject
        chosen = rng.choice(good) if good else bug['fixed']   # sft pile -> gold fix
        pairs.append({'id': bid, 'pile': routing.get(bid),
                      'prompt': build_repair_prompt(bug['buggy'], bug['test_list']),
                      'chosen': '```python\n' + chosen.strip() + '\n```',
                      'rejected': '```python\n' + rng.choice(bad).strip() + '\n```'})
        done += 1
        if done % 100 == 0:
            print(f'{done}/{len(samples)} bugs graded, {len(pairs)} pairs')
    json.dump(pairs, open(PAIRS_PATH, 'w'))
from collections import Counter
print(Counter(p['pile'] for p in pairs), '| total pairs:', len(pairs))
cl = sum(len(p['chosen']) for p in pairs) / len(pairs)
rl_ = sum(len(p['rejected']) for p in pairs) / len(pairs)
print(f'LENGTH-BIAS CHECK: mean chosen {cl:.0f} chars vs rejected {rl_:.0f} chars')
print('(if chosen is systematically much longer/shorter, DPO may learn length, not correctness)')

In [ ]:
# Shared eval helpers (same protocol as notebook 05)
import torch
from unsloth import FastLanguageModel

@torch.no_grad()
def generate_k(model, tok, source_code, test_list, k):
    text = tok.apply_chat_template(
        [{'role': 'user', 'content': build_repair_prompt(source_code, test_list)}],
        tokenize=False, add_generation_prompt=True)
    enc = tok([text], return_tensors='pt', padding=True, padding_side='left').to('cuda')
    out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                         num_return_sequences=k, max_new_tokens=512,
                         pad_token_id=tok.eos_token_id)
    gen = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [extract_code(t) for t in gen]

def dev_eval(model, tok, tag, k=16):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for b in dev_bugs:
        codes = generate_k(model, tok, b['buggy'], b['test_list'], k)
        with ThreadPoolExecutor(max_workers=4) as pool:
            flags = list(pool.map(lambda c: passes(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    p16 = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    res = {'tag': tag, 'pass@1': p1, 'pass@16': p16, 'gap': p16 - p1, 'per_bug': per_bug}
    json.dump(res, open(f'{PHASE4}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@16={p16*100:.1f}%  gap={(p16-p1)*100:.1f}")
    return res

def restraint_probe(model, tok, tag, k=4):
    FastLanguageModel.for_inference(model)
    norm = lambda s: re.sub(r'\s+', ' ', s.strip())
    still_pass = unchanged = total = 0
    for r in dev_clean:
        codes = generate_k(model, tok, r['code'], r['test_list'], k)
        with ThreadPoolExecutor(max_workers=4) as pool:
            flags = list(pool.map(lambda c: passes(c, r), codes))
        still_pass += sum(flags)
        unchanged += sum(norm(c) == norm(r['code']) for c in codes)
        total += k
    json.dump({'tag': tag, 'still_pass': still_pass/total, 'unchanged': unchanged/total},
              open(f'{PHASE4}/restraint_probe_{tag}.json', 'w'))
    print(f"[probe {tag}]  still passes: {still_pass/total*100:.1f}%  unchanged: {unchanged/total*100:.1f}%")

In [ ]:
# DPO per seed (A3 paired lineage): merge SFT-seed -> fresh LoRA -> DPO vs SFT anchor
from unsloth import PatchDPOTrainer
PatchDPOTrainer()
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

def run_seed(seed):
    print(f'===== DPO SEED {seed} (init: sft_notrace_s{seed}_ep1) =====')
    merged = f'/content/merged_s{seed}'
    if not os.path.exists(merged):
        m, t = FastLanguageModel.from_pretrained(
            f'{PHASE3}/sft_notrace_s{seed}_ep1', max_seq_length=1024,
            load_in_4bit=False, dtype=torch.bfloat16)
        m.save_pretrained_merged(merged, t, save_method='merged_16bit')
        del m, t; gc.collect(); torch.cuda.empty_cache()
        print('merged SFT weights ->', merged)
    model, tok = FastLanguageModel.from_pretrained(
        merged, max_seq_length=1024, load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=32, lora_alpha=64, lora_dropout=0,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=seed)
    def to_row(p):
        return {'prompt': tok.apply_chat_template([{'role': 'user', 'content': p['prompt']}],
                                                  tokenize=False, add_generation_prompt=True),
                'chosen': p['chosen'], 'rejected': p['rejected']}
    ds = Dataset.from_list([to_row(p) for p in pairs]).shuffle(seed=seed)
    FastLanguageModel.for_training(model)
    trainer = DPOTrainer(model=model, ref_model=None, tokenizer=tok, train_dataset=ds,
        args=DPOConfig(per_device_train_batch_size=4, gradient_accumulation_steps=4,
            num_train_epochs=1, learning_rate=1e-5, lr_scheduler_type='cosine',
            warmup_ratio=0.1, beta=0.1, max_length=1024, max_prompt_length=768,
            logging_steps=5, seed=seed, output_dir='/content/out',
            report_to='none', save_strategy='no'))
    trainer.train()
    model.save_pretrained(f'{PHASE4}/dpo_s{seed}_ep1')
    dev_eval(model, tok, f'dpo_ep1_seed{seed}')
    restraint_probe(model, tok, f'dpo_s{seed}')
    del trainer, model, tok
    gc.collect(); torch.cuda.empty_cache()

for s in (3407, 42, 1234):
    if os.path.exists(f'{PHASE4}/dev_eval_dpo_ep1_seed{s}.json'):
        print(f'seed {s} already done, skipping'); continue
    run_seed(s)

In [ ]:
# SFT vs DPO on the dev slice (ruler: pass@1 spread 1.5 pts / sd 0.8)
import statistics as st
SEEDS = (3407, 42, 1234)
print(f"{'tag':<18} pass@1   pass@16   gap")
sft_rows, dpo_rows = [], []
for s in SEEDS:
    r = json.load(open(f'{PHASE3}/dev_eval_ep1_seed{s}.json')); sft_rows.append(r)
    print(f"sft_seed{s:<10} {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%  {r['gap']*100:5.1f}")
for s in SEEDS:
    p = f'{PHASE4}/dev_eval_dpo_ep1_seed{s}.json'
    if not os.path.exists(p): continue
    r = json.load(open(p)); dpo_rows.append(r)
    print(f"dpo_seed{s:<10} {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%  {r['gap']*100:5.1f}")
if len(dpo_rows) == 3:
    for name, rows in (('SFT', sft_rows), ('DPO', dpo_rows)):
        v = [r['pass@1']*100 for r in rows]
        print(f"{name}: mean pass@1 {st.mean(v):.1f}%  sd {st.stdev(v):.1f}")
    d = st.mean([r['pass@1'] for r in dpo_rows]) - st.mean([r['pass@1'] for r in sft_rows])
    print(f"\nDPO - SFT mean pass@1: {d*100:+.1f} pts (ruler: differences < ~1.5 pts = tie)")
    print('Also watch pass@16: DPO squeezes — some gap-eating is expected, collapse is not.')
print('\nRestraint probes:')
for s in SEEDS:
    p = f'{PHASE4}/restraint_probe_dpo_s{s}.json'
    if os.path.exists(p):
        r = json.load(open(p))
        print(f"  dpo_s{s}: still-passes {r['still_pass']*100:.1f}%  unchanged {r['unchanged']*100:.1f}%  (sft was ~68-75 / ~28-32)")
print('\nBring the whole printout back to the session.')

## Bring back to the session
1. The **pair-mining report** (counts by pile + the length-bias check line)
2. The **training-loss curves** if anything looked odd (DPO loss should start
   ~0.69 = log 2 and drift down; instant ~0 means something is broken)
3. The **SFT vs DPO table** and the **restraint probe lines**

After this: the DPO milestone gets folded into the Phase-7 final exam (one
held-out run per arm), and Phase 5 — GRPO, the main event — begins.